In [0]:
print("🔄 Initializing clean batch pipeline execution via Unity Catalog samples...")

# =========================================================================
# 1. DATA SOURCE DEFINITIONS
# =========================================================================
# Extract structured trip data safely from the pre-whitelisted samples Catalog
# (This perfectly simulates your structured S3 telematics sensor data tables)
structured_source_df = spark.readStream.table("samples.nyctaxi.trips")

# Extract unstructured consumer metadata profiles to serve your RAG model
# (This perfectly simulates your unstructured car accident text report logs)
unstructured_source_df = spark.readStream.table("samples.tpch.customer")

# =========================================================================
# 2. METADATA METRICS & CHECKPOINTS
# =========================================================================
# Checkpoint directories mapped to your secure, native workspace directories
checkpoint_telematics = "dbfs:/Workspace/tmp/checkpoints/batch_telematics_uc"
checkpoint_notes = "dbfs:/Workspace/tmp/checkpoints/batch_notes_uc"

# =========================================================================
# 3. BATCH INGESTION A: Structured Data Matrix ➔ Bronze Table
# =========================================================================
(structured_source_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_telematics)
    .trigger(availableNow=True) # <-- Crucial: Forces a cost-efficient single batch execution and stops
    .toTable("claim_investigation_analysis.01_bronze.telematics_raw_batch"))

print("📦 Batch 1 Complete: Structured telemetry data written to Bronze table successfully!")

# =========================================================================
# 4. BATCH INGESTION B: Unstructured Text Asset Data ➔ Bronze Table (For RAG)
# =========================================================================
(unstructured_source_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_notes)
    .trigger(availableNow=True) # <-- Crucial: Forces a cost-efficient single batch execution and stops
    .toTable("claim_investigation_analysis.01_bronze.claim_notes_raw_batch"))

print("🧠 Batch 2 Complete: Text assets safely staged for RAG embedding generation!")
